# Session 7 Homework · Which Ingredient Sets a House Price?

**Expected time:** 25–35 minutes · **Bring questions** — next session opens with homework review.

In class you ranked the habits behind a test score. Now rank the ingredients behind a **house price** — and use the same fair-comparison trick.

**Done means:**

- a multi-feature model whose **cost is lower** than an area-only model
- a **raw** and a **scaled** coefficient table
- an insight naming the top feature (by the **scaled** table) and reading one **negative** coefficient

This one needs light prep: two columns have missing values. A single feature avoided them last time; using five, we must fix them first.

## Step 1 · Prep and fit

Fill the missing values, then predict `price_lakhs` from five numeric features. (We skip the text column `neighborhood_type` today to keep the focus on coefficients.)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

homes = pd.read_csv("../../../datasets/secondary/housing.csv")

# Fill the two columns with missing values (Session 2 move).
homes["age_years"] = homes["age_years"].fillna(homes["age_years"].median())
homes["distance_to_center_km"] = homes["distance_to_center_km"].fillna(homes["distance_to_center_km"].median())
print("missing values remaining:", homes.isna().sum().sum())

features = ["area_sqft", "bedrooms", "bathrooms", "age_years", "distance_to_center_km"]
X = homes[features]
y = homes["price_lakhs"]

def cost(y_true, y_pred):
    return np.mean((y_true - y_pred)**2)

model = LinearRegression().fit(X, y)
area_only = LinearRegression().fit(homes[["area_sqft"]], y)
print(f"cost with area only:      {cost(y, area_only.predict(homes[['area_sqft']])):.1f}")
print(f"cost with five features:  {cost(y, model.predict(X)):.1f}")

### ✏️ Did more features help?

Is the five-feature cost lower than area-only? What does that tell you?

*Your answer:* Yes — the five-feature cost is lower. Price depends on more than size, so extra ingredients let the line fit the houses better.

## Step 2 · Raw coefficients (the misleading view)

In [ ]:
raw = pd.DataFrame({"feature": features, "coefficient": model.coef_})
raw = raw.sort_values("coefficient", key=lambda c: c.abs(), ascending=False)
raw.round(3)

These are in different units (square feet vs number of bedrooms vs years), so their sizes aren't a fair race — exactly like the habits in class.

## Step 3 · Scale, refit, and rank fairly

In [ ]:
from sklearn.preprocessing import StandardScaler

X_scaled = StandardScaler().fit_transform(X)
scaled_model = LinearRegression().fit(X_scaled, y)

# Scaling shouldn't change the cost — check.
print(f"cost unscaled: {cost(y, model.predict(X)):.1f}")
print(f"cost scaled:   {cost(y, scaled_model.predict(X_scaled)):.1f}")

scaled = pd.DataFrame({"feature": features, "scaled_coefficient": scaled_model.coef_})
scaled = scaled.sort_values("scaled_coefficient", key=lambda c: c.abs(), ascending=False)
scaled.round(2)

### ✏️ Write the insight

1. Using the **scaled** table: which feature is most associated with a higher price? (Start *"In this data, …"*)
2. Name one feature with a **negative** coefficient and give a plausible real-world reason it would pull price down.

*Your answers:*

1. In this data, area (square feet) is the feature most associated with a higher price — it has the largest scaled coefficient.
2. `age_years` (and often `distance_to_center_km`) has a negative coefficient: older homes, and homes farther from the centre, are associated with lower prices — which matches how people actually value houses.

## One last reflection

✏️ The housing ranking is messier and more arguable than the clean student-habits ranking. The cost dropped nicely — but could you tell a parent, in one number, *how good* this price model is? What would that number even be?

*Your answer:* Not really yet — "cost = 26" or "726" isn't a number anyone outside this class would understand, and it's in squared units. We need a quality score in plain rupees or on a 0–1 scale. That's exactly Session 8: MAE, RMSE, and R², measured on data the model has never seen.